# Maximum likelihood: an isotropic bivariate Gaussian

Can you fit a Gaussian to a cloud of points by eye—and then improve the fit using likelihood?

**Run all cells**, then move the sliders for the mean coordinates $\mu_x,\mu_y$ and the variance $\sigma^2$. The plot and likelihood update when you release a slider. The points stay fixed while you explore. Try to maximize the **log-likelihood** before clicking **Show MLE**.

An isotropic Gaussian has the same variance in every direction:

$$\mathbf{x}_i \sim \mathcal{N}(\boldsymbol\mu,\sigma^2 I_2), \qquad
\boldsymbol\mu=\begin{pmatrix}\mu_x\\mu_y\end{pmatrix}, \qquad
\Sigma=\begin{pmatrix}\sigma^2&\ldots&\sigma^2\end{pmatrix}.$$

Its equal-density contours are circles. Changing $\sigma^2$ changes their size; changing the mean moves their center. This notebook uses NumPy, Matplotlib, and ipywidgets in Jupyter Notebook or JupyterLab.

In [4]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from matplotlib.lines import Line2D
import ipywidgets as widgets
from IPython.display import display, Markdown

# Change these settings and re-run all cells to explore another point cloud.
seed = 42
n = 60
true_mean = np.array([1.5, -0.5])
true_variance = 1.2
rng = np.random.default_rng(seed)
points = rng.normal(loc=true_mean, scale=np.sqrt(true_variance), size=(n, 2))


## Scoring a fit

Assuming independent observations, the likelihood is the product of the densities at the observed points:

$$L(\boldsymbol\mu,\sigma^2)=\prod_{i=1}^n \frac{1}{2\pi\sigma^2}
\exp\!\left(-\frac{\|\mathbf{x}_i-\boldsymbol\mu\|^2}{2\sigma^2}\right).$$

Products can be extremely small, so we compute their logarithm instead:

$$\ell(\boldsymbol\mu,\sigma^2)=-n\log(2\pi\sigma^2)
-\frac{1}{2\sigma^2}\sum_{i=1}^n\|\mathbf{x}_i-\boldsymbol\mu\|^2.$$

Maximizing $\ell$ gives the same fit as maximizing $L$. **Higher is better**, including when the values are negative. Likelihood is a product of *densities*, not the probability of observing these exact continuous-valued points, and it need not be between zero and one.

In [5]:
def log_likelihood(data, mean, variance):
    """Joint log-density for independent 2D points with covariance variance * I."""
    data = np.asarray(data, dtype=float)
    mean = np.asarray(mean, dtype=float)
    if data.ndim != 2 or data.shape[1] != 2 or len(data) < 1 or not np.isfinite(data).all():
        raise ValueError("data must be a finite, nonempty (n, 2) array.")
    if mean.shape != (2,) or not np.isfinite(mean).all():
        raise ValueError("mean must contain two finite coordinates.")
    if not np.isfinite(variance) or variance <= 0:
        raise ValueError("variance must be positive and finite.")
    squared_distances = np.sum((data - mean) ** 2)
    return float(-len(data) * np.log(2 * np.pi * variance)
                 - squared_distances / (2 * variance))


def scientific_likelihood(log_value):
    """Format exp(log_value) without underflowing the likelihood to zero."""
    log10_value = log_value / np.log(10)
    exponent = int(np.floor(log10_value))
    mantissa = 10 ** (log10_value - exponent)
    return f"{mantissa:.4f} × 10^({exponent})"


mle_mean = points.mean(axis=0)
mle_variance = float(np.sum((points - mle_mean) ** 2) / (2 * len(points)))
if mle_variance <= 0:
    raise ValueError("Use a point cloud with nonzero spread for a finite positive-variance MLE.")
mle_log_likelihood = log_likelihood(points, mle_mean, mle_variance)


In [6]:
# Fixed, square axes make changes in spread easy to compare.
plot_low = float(min(points.min(), 0) - 2)
plot_high = float(max(points.max(), 0) + 2)
slider_style = {"description_width": "100px"}
slider_layout = widgets.Layout(width="550px")
mean_x = widgets.FloatSlider(
    value=0, min=plot_low, max=plot_high, step=0.05, description="Mean μx:",
    continuous_update=False, readout_format=".3f", style=slider_style, layout=slider_layout,
)
mean_y = widgets.FloatSlider(
    value=0, min=plot_low, max=plot_high, step=0.05, description="Mean μy:",
    continuous_update=False, readout_format=".3f", style=slider_style, layout=slider_layout,
)
variance_control = widgets.FloatLogSlider(
    value=2 * mle_variance, base=10, min=np.log10(mle_variance / 50),
    max=np.log10(mle_variance * 20), step=0.02, description="Variance σ²:",
    continuous_update=False, readout_format=".4f", style=slider_style, layout=slider_layout,
)
show_mle = widgets.Checkbox(value=False, description="Show MLE")
fit_button = widgets.Button(description="Set sliders to MLE", button_style="info", disabled=True)
output = widgets.Output()
updating = False


def render_fit(change=None):
    if updating:
        return
    mean = np.array([mean_x.value, mean_y.value])
    variance = variance_control.value
    sigma = np.sqrt(variance)
    ll = log_likelihood(points, mean, variance)
    with output:
        output.clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(7, 6))
        ax.scatter(points[:, 0], points[:, 1], color="#304B63", s=28, alpha=0.75,
                   label=f"Observed points (n={len(points)})", zorder=3)
        ax.scatter(*mean, color="#C65319", marker="+", s=180, linewidths=2.5, zorder=5)
        colors = ["#C65319", "#DC8A39", "#E6B86A"]
        handles = [ax.collections[0], Line2D([], [], color="#C65319", marker="+",
                   linestyle="none", markersize=12, label="Chosen mean")]
        for k, color in zip([1, 2, 3], colors):
            ax.add_patch(Circle(mean, k * sigma, fill=False, edgecolor=color, linewidth=2))
            handles.append(Line2D([], [], color=color, linewidth=2, label=f"Radius {k}σ"))
        if show_mle.value:
            ax.scatter(*mle_mean, color="#287D63", marker="x", s=90, zorder=5)
            for k in [1, 2, 3]:
                ax.add_patch(Circle(mle_mean, k * np.sqrt(mle_variance), fill=False,
                                    edgecolor="#287D63", linestyle="--", linewidth=1.4))
            handles.append(Line2D([], [], color="#287D63", linestyle="--", label="MLE contours"))
        ax.set(xlim=(plot_low, plot_high), ylim=(plot_low, plot_high),
               xlabel="x", ylabel="y", title="Fit the Gaussian to the fixed point cloud")
        ax.set_aspect("equal", adjustable="box")
        ax.grid(alpha=0.15)
        ax.legend(handles=handles, loc="upper left", fontsize=9)
        fig.tight_layout()
        plt.show()
        plt.close(fig)
        display(Markdown(
            f"**Log-likelihood: {ll:.4f}** &nbsp; | &nbsp; "
            f"Likelihood: `{scientific_likelihood(ll)}`\n\n"
            f"Mean = ({mean[0]:.4f}, {mean[1]:.4f}); "
            f"covariance = diag({variance:.4f}, {variance:.4f}); σ = {sigma:.4f}."
        ))
        if show_mle.value:
            gap = max(0.0, mle_log_likelihood - ll)
            display(Markdown(
                f"**MLE:** μ = ({mle_mean[0]:.4f}, {mle_mean[1]:.4f}), "
                f"σ² = {mle_variance:.4f}. Maximum log-likelihood = {mle_log_likelihood:.4f}.\n\n"
                f"**Log-likelihood gap to the maximum: {gap:.4f}** (zero is best)."
            ))


def toggle_mle(change):
    fit_button.disabled = not change["new"]
    render_fit()


def set_to_mle(_):
    global updating
    updating = True
    try:
        mean_x.value, mean_y.value = map(float, mle_mean)
        variance_control.value = mle_variance
    finally:
        updating = False
    render_fit()


for control in [mean_x, mean_y, variance_control]:
    control.observe(render_fit, names="value")
show_mle.observe(toggle_mle, names="value")
fit_button.on_click(set_to_mle)
display(widgets.VBox([
    mean_x, mean_y, variance_control,
    widgets.HBox([show_mle, fit_button]),
    widgets.HTML("Contours are at radii σ, 2σ, and 3σ; large circles may extend outside the fixed view."),
    output,
]))
render_fit()
